# Phase 1 — Parse Documents and Preserve Evidence

**Roadmap reference:** `ContractIQ_Project_Execution_Roadmap.md`, Phase 1

**Goal:** parse real DOCX contracts into paragraph-level clause records and table records, each with a stable, re-locatable source location (`paragraph:14`, `table:2,row:4,cell:3`).

**Guiding rule:** start with ONE representative MSA and ONE table-heavy supporting document. Only run the full 36-document batch after manually reviewing these.

**Completion gate:** every extracted clause can be located in the original document; tables preserve real cell values; failures are recorded per file, not swallowed; no source document is modified.

## 1. Load configuration and the Phase 0 manifest

In [1]:
with open("../src/contractiq/parsing.py", encoding="utf-8") as f:
    content = f.read()

for fn in ["run_phase1_sample", "run_phase1_full", "parse_docx_document", "load_parsed_document"]:
    print(fn, "->", "FOUND" if f"def {fn}" in content else "MISSING")

print(f"\nTotal file length: {len(content)} chars")

run_phase1_sample -> FOUND
run_phase1_full -> FOUND
parse_docx_document -> FOUND
load_parsed_document -> FOUND

Total file length: 10211 chars


In [2]:
import sys
sys.path.append("..")

from src.contractiq import config
from src.contractiq.ingestion import load_manifest
from src.contractiq.parsing import (
    run_phase1_sample,
    run_phase1_full,
    load_parsed_document,
    table_cell_location,
)


manifest = load_manifest()
print(f"Loaded manifest: {len(manifest)} documents")

if len(manifest) != config.EXPECTED_TOTAL_DOCUMENTS:
    print(f"⚠ Expected {config.EXPECTED_TOTAL_DOCUMENTS}, got {len(manifest)}. "
          "Re-run 00_Project_Setup.ipynb before continuing.")

Loaded manifest: 36 documents


## 2. Pick one representative MSA and one table-heavy supporting document

Look at the printed manifest below and choose two real `doc_id` values — one from `master_agreements` (an MSA) and one that's likely to contain tables (e.g. a rate card in `commercial_docs`, or an invoice in `financial_billing`). Update `SAMPLE_DOC_IDS` in the next cell with the actual IDs you pick.

In [3]:
for r in manifest:
    if r.category_key in ("master_agreements", "commercial_docs", "financial_billing"):
        print(f"  {r.doc_id:<10} {r.category_key:<20} {r.filename}")

  COM-001    commercial_docs      Pricing table Sample 1.docx
  COM-002    commercial_docs      Pricing table Sample 2.docx
  FIN-001    financial_billing    Billing schedule sample 1.docx
  FIN-002    financial_billing    Billing schedule sample 2.docx
  FIN-003    financial_billing    Invoices and credit note sample 1.docx
  FIN-004    financial_billing    Invoices and credit notes sample 2.docx
  MLA-001    master_agreements    MSA Sample 1.docx
  MLA-002    master_agreements    MSA sample 2.docx
  MLA-003    master_agreements    NDA sample 1.docx
  MLA-004    master_agreements    NDA sample 2.docx
  MLA-005    master_agreements    RATE CARD AGREEMENT Sample 2.docx
  MLA-006    master_agreements    RATE CARD AGREEMENTSample 1.docx


## 3. Parse the sample — manual verification step

This does NOT touch the other 34 documents yet. Review the printed clause/table counts and the location-verification result before moving on.

In [4]:
# TODO: replace with the two doc_ids you picked above
SAMPLE_DOC_IDS = ["MLA-001", "COM-001"]

sample_results = run_phase1_sample(manifest, SAMPLE_DOC_IDS)

  MLA-001 (MSA Sample 1.docx): ParseStatus.PARSED — 34 clauses, 0 tables
      ✓ every clause location verified against the source
  COM-001 (Pricing table Sample 1.docx): ParseStatus.PARSED — 38 clauses, 2 tables
      ✓ every clause location verified against the source


## 4. Manually inspect a few extracted clauses

Read these against the actual Word document open side-by-side. This is the human-verification step the roadmap requires before scaling up — not optional, and not replaceable by the automated check above alone.

In [10]:
sample_id = SAMPLE_DOC_IDS[1]
parsed = load_parsed_document(sample_id)

print(f"First 5 clauses of {sample_id}:\n")
for clause in parsed["clauses"][:5]:
    print(f"[{clause['source_location']['locator']}] "
          f"heading={clause['heading']!r} hint={clause['heading_hint']!r}")
    print(f"    {clause['text'][:120]}")
    print()

First 5 clauses of COM-001:

[paragraph:0] heading=None hint=None
    Sample 1: Pricing Table Agreement

[paragraph:1] heading=None hint=None
    Between: Primevolta Hardware Solutions (Vendor)
And: FinSecure Services (Client)
Date: 11 December 2025

[paragraph:3] heading=None hint=None
    1. Introduction

[paragraph:4] heading=None hint=None
    This Pricing Table Agreement (“Agreement”) is entered into between Primevolta Hardware Solutions, having its principal o

[paragraph:5] heading=None hint=None
    The purpose of this Agreement is to ensure transparency, consistency, and mutual understanding regarding the pricing of 



In [11]:
print(f"Tables found in {sample_id}: {len(parsed['tables'])}\n")
if parsed["tables"]:
    first_table = parsed["tables"][0]
    print(f"Table 0 has {len(first_table['rows'])} rows:")
    for row in first_table["rows"][:5]:
        print("   ", row)

Tables found in COM-001: 2

Table 0 has 6 rows:
    ['Product Category', 'Product Description', 'Unit Price (INR)', 'Volume Tier 1 (1–10 units)', 'Volume Tier 2 (11–50 units)', 'Volume Tier 3 (51+ units)', 'Notes']
    ['Servers', 'Rack Server X100', '1,25,000', '1,25,000', '1,20,000', '1,15,000', 'Includes 3-year warranty']
    ['Networking Equipment', 'Core Router R500', '75,000', '75,000', '72,500', '70,000', 'Annual maintenance optional']
    ['Storage Solutions', 'SAN Storage S2000', '2,00,000', '2,00,000', '1,95,000', '1,90,000', 'Scalable up to 500TB']
    ['Peripheral Devices', 'UPS 10kVA', '50,000', '50,000', '48,000', '46,000', 'Includes installation']


## 5. Only after the sample looks correct: parse the full manifest

Uncomment and run once you've manually confirmed step 4 is accurate. This processes all 36 documents; failures are reported per file and don't stop the batch.

In [12]:
full_results = run_phase1_full(manifest)

Parsed 36 document(s): 2047 clauses, 30 tables total.


---
### Next: `02_Entity_Clause_and_EDA.ipynb`

Extract parties, dates, monetary amounts, and contract-specific terms from these clause records, keeping raw text and normalized values separate and source-linked.